# Session 13 · Case study: error analysis of 20 misclassified decisions

**Block 2 worked example.** Analyse 20 misclassified decisions with the help of their English keywords and the English heading texts, look at the most informative n-grams, measure the leak through the customs' justification, and use what you find to improve the classifier.

Run time: about 6 minutes.

Theory: [03-lecture-text-classification-and-error-analysis.ipynb](03-lecture-text-classification-and-error-analysis.ipynb).

*Author: course team, licence CC-BY-4.0.*

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
decisions = pd.read_parquet(DATA / "train_sample.parquet")
nomenclature = pd.read_parquet(DATA / "nomenclature.parquet").set_index("heading")
texts = decisions["description"]                     # one document = one description of goods
print(decisions.shape, decisions["heading"].nunique(), "headings")

(50000, 14) 934 headings


In [2]:
year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]   # validation by time, as on the leaderboard
print(len(train), "training decisions (2017-2021),", len(valid), "validation decisions (2022-2023)")

36801 training decisions (2017-2021), 13199 validation decisions (2022-2023)


In [3]:
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline


def make_model(**tfidf_kw):
    kw = dict(min_df=2, sublinear_tf=True) | tfidf_kw
    return make_pipeline(TfidfVectorizer(**kw),
                         SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None, random_state=0, n_jobs=-1))


model = make_model().fit(train["description"], train["heading"])
pred = model.predict(valid["description"])
print("accuracy", round(accuracy_score(valid["heading"], pred), 3),
      "macro-F1", round(f1_score(valid["heading"], pred, average="macro"), 3))

accuracy 0.772 macro-F1 0.511


## 1. Where are the errors?

In [4]:
val = valid.assign(pred=pred)
wrong = val[val["heading"] != val["pred"]]
print(len(wrong), "errors; share inside the right chapter:",
      round((wrong["heading"].str[:2] == wrong["pred"].str[:2]).mean(), 2))
top_pairs = wrong[["heading", "pred"]].value_counts().head(10).rename("n").reset_index()
top_pairs["true_name"] = top_pairs["heading"].map(nomenclature["heading_description"]).str[:45]
top_pairs["pred_name"] = top_pairs["pred"].map(nomenclature["heading_description"]).str[:45]
top_pairs

3003 errors; share inside the right chapter: 0.3


,heading,pred,n,true_name,pred_name
0,8524,8548,17,"Flat panel display modules, whether or not in",Electrical parts of machinery or apparatus; n
1,8539,9405,13,"Lamps; electric filament or discharge lamps,",Luminaires and light fittings; including sear
2,8524,8529,12,"Flat panel display modules, whether or not in",Transmission apparatus; parts suitable for us
3,8708,3926,11,"Motor vehicles; parts and accessories, of hea",Articles of plastics and articles of other ma
4,3924,3926,11,"Tableware, kitchenware, other household artic",Articles of plastics and articles of other ma
5,7308,7326,11,Structures of iron or steel and parts thereof,"Iron or steel; articles, n.e.c. in chapter 73"
6,6404,6405,10,"Footwear; with outer soles of rubber, plastic",Footwear; other footwear n.e.c. in chapter 64
7,8806,8525,9,Unmanned aircraft,Transmission apparatus for radio-broadcasting
8,3926,9503,9,Articles of plastics and articles of other ma,"Tricycles, scooters, pedal cars and similar w"
9,8536,8544,8,"Electrical apparatus for switching, protectin","Insulated wire, cable and other electric cond"


**Question.** Headings 8524 and 8548 are confused most often. Look up 8524 in the nomenclature: when was it created? (Hint: HS 2022.)

## 2. Draw 20 errors and categorise them

The seed is fixed so that everyone in the class reads the same decisions. Read each one with its keywords and the two heading texts.

In [5]:
pd.set_option("display.max_colwidth", 250)
errors20 = wrong.sample(20, random_state=1).copy()
errors20["true_heading"] = errors20["heading"] + " " + errors20["heading"].map(nomenclature["heading_description"]).str[:60]
errors20["predicted"] = errors20["pred"] + " " + errors20["pred"].map(nomenclature["heading_description"]).str[:60]
errors20["text"] = errors20["description"].str.replace(r"\s+", " ", regex=True).str[:250]
errors20[["language", "true_heading", "predicted", "keywords", "text"]]

,language,true_heading,predicted,keywords,text
41846,nl,"9605 Travel sets; for personal toilet, sewing, shoe or clothes cl","3301 Oils; essential (concretes, absolutes); concentrates thereof","AEROPLANES,BOTTLES,EMPTY,FOR FILLING,OF PLASTICS,SPATULAS,TRANSPARENT,TRAVEL SETS","Een zogenoemd reisassortiment met –volgens opgave- onder andere de volgende samenstelling en kenmerken: - om te vullen met shampoo, zeep en andere cosmetische producten voor op reis; - 2 kunststof flesjes met schroefdop, met een inhoud van 100 ml..."
45577,fr,9027 Instruments and apparatus; for physical or chemical analysis,8543 Electrical machines and apparatus; having individual functio,"ACIDS,AS MODULE,CHROMOTOGRAPHS,FOR MEDICAL USE,FOR PHARMACEUTICALS,PUMPS,SCIENTIFIC AND MEASURING EQUIPMENT","Module destiné à être installé dans un environnement industriel (équipé d’enceintes blindées) pour permettre d’isoler par chromatographie un élément (Lutétium-177), utilisé en médecine nucléaire. Le module permet la production d’une solution (de ..."
43912,nl,"8214 Cutlery; other articles, (e.g. hair clippers, butchers' or k","9503 Tricycles, scooters, pedal cars and similar wheeled toys; do","ART. OF CUTLERY, CUTTING UTENSILS,HOLDERS,KNIVES,OF BASE METALS,OF PLASTICS","Een zogenoemd foliemes met –volgens opgave– onder andere de volgende kenmerken: - te gebruiken bij het verwijderen van folie, spanbanden, tape en dergelijke van dozen en pallets; - van onedel metaal; - een behuizing van kunststof, inclusief verwi..."
43781,nl,"8421 Centrifuges, including centrifugal dryers; filtering or puri",3926 Articles of plastics and articles of other materials of head,"AS CYLINDERS,CHROMOTOGRAPHS,CONNECTIONS,FILTERS,FOR BUILDING-IN,LIQUID FILTERS","Een filterkolom met -volgens opgave- onder andere de volgende kenmerken en eigenschappen: - voor gebruik in de chromatografie, om individuele componenten af te scheiden van een vloeistofmonster; - geschikt voor plaatsing in elke vloeistofchromato..."
38644,cs,"4419 Tableware and kitchenware, of wood","7318 Screws, bolts, nuts, coach screws, screw hooks, rivets, cott","ARTICLES OF WOOD,BIRCH,CUTLERY, TABLEWARE,DISPOSABLE,MACHINE-WORKED,NATURAL COLOURED,OF WOOD,SPOONS,SURFACE WORKED,WOOD","Dle deklarovaných údajů se jedná o výrobek ze dřeva – nápojové míchátko. Plochá, dlouhá, na obou koncích zaoblená tyčinka z březového dřeva je určena k míchání nápojů, např. kávy. Určeno v gastronomii k jednorázovému použití a k přímému styku s n..."
48433,cs,"2404 Products containing tobacco, reconstituted tobacco, nicotine",3824 Prepared binders for foundry moulds or cores; chemical produ,"BILBERRIES,HERBS,MINT,NICOTINE,THYME,WITHOUT TOBACCO","Dle deklarovaných údajů a přiložené fotodokumentace se jedná o směs ke kouření, s obsahem nikotinu, neobsahující tabák. Směs je naplněna do tyčinek, které se vkládají do HNB přístroje a je určena k zahřívání (vdechování bez spalování). Složení: č..."
41515,fr,4421 Wooden articles n.e.c. in heading no. 4414 to 4420,"9403 Furniture and parts thereof, n.e.c. in chapter 94","ARTICLES OF WOOD,FOR HYGIENIC USE,LAVATORY COVERS,LAVATORY PANS,LAVATORY SEATS,NON-TROPICAL WOODS,OF STEEL,SANITARY WARE,SCREWS,SPRUCE TREE","Article composite, de type appareil sanitaire, pour usage hygiénique, se présentant sous la forme de toilettes sèches en bois épicéa (97%). L’article comprend un corps, un plateau, et une lunette-abattant en bois. Les charnières du plateau et de ..."
42080,fr,"7013 Glassware of a kind used for table, kitchen, toilet, office,","9405 Luminaires and light fittings; including searchlights, spotl","BELL JARS,BY STORAGE BATTERIES,FOR DECORATION,GATHERED MECHANICALLY,GLASS,LED,MISCELLANEOUS ARTICLES,ORNAMENTAL ARTICLES,WITH LUMINOUS DEVICE,WITH SOCLE","Article de décoration intérieure, se présentant sous la forme d’une goutte en verre dotée d’un socle et d’une accroche avec ruban, comprenant des personnages Disney en résine en son intérieur, ainsi qu’un éclairage par LED. Dimensions : l 83 x L ..."
44630,e

Give each decision **one** category. Suggested categories (add your own):

- `borderline`: related headings; the legal notes decide
- `material-function`: the model follows the material, the tariff the function (or the reverse)
- `language`: rare language or few training examples
- `technical`: chemical or technical knowledge needed
- `set`: sets and composite goods
- `new-heading`: heading created or changed in HS 2022
- `unclear`: no clear reason

In [6]:
# fill in one category per decision, in the order of the table above
my_categories = [""] * 20
errors20["category"] = my_categories
errors20["category"].value_counts()

category
    20
Name: count, dtype: int64

## 3. What does the model rely on?

In [7]:
vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["sgdclassifier"]
names = vec.get_feature_names_out()
top = {h: list(names[np.argsort(clf.coef_[list(clf.classes_).index(h)])[::-1][:12]])
       for h in ["6403", "6404", "9503", "3926", "8517"]}
pd.DataFrame(top)

,6403,6404,9503,3926,8517
0,cuir,kunststoff,jouet,3926,appel
1,6403,textile,spielzeug,ouvrage,kommunikation
2,24,1910,speelgoed,plastique,mhz
3,leder,caoutchouc,leksak,statuetten,réseau
4,1200,1100,jouets,andere,communication
5,leather,sole,enfants,műanyagból,receive
6,naturel,spinnstoffen,toy,bâche,töne
7,24cm,6404,children,kunststoffen,émission
8,more,chaussures,vermaak,matière,berendezés
9,antidérapante,oberteil,puzzle,cet,telefoon


**Question.** Several of the top features are numbers such as `6403`. Count how often a description quotes its own heading, and how accurate the model is with and without such a number.

In [8]:
has_number = pd.Series([bool(re.search(rf"(?<!\d){h}(?!\d)", t))
                        for h, t in zip(valid["heading"], valid["description"])], index=valid.index)
print("share quoting the heading:", round(has_number.mean(), 3))
val.groupby(has_number).apply(lambda g: pd.Series({"n": len(g), "accuracy": (g["heading"] == g["pred"]).mean()})).round(3)

share quoting the heading: 0.246


,n,accuracy
False,9956.0,0.705
True,3243.0,0.980


## 4. A column that exists only after the decision

Reading the keywords and the justification helped us understand the errors. As **model inputs** they are forbidden: customs writes them together with the decision, so a new request does not have them. How large is the temptation? Train the same model once on the description and once on description + justification, validate both with a random split inside 2017–2021 (as a careless notebook would), and then apply both where they will be used: to the 2022–2023 descriptions alone.

In [9]:
from sklearn.model_selection import train_test_split

just = decisions["classification_justification"].fillna("")
print("justification names the heading:", round(np.mean([h in j for h, j in zip(decisions["heading"], just)]), 2))
first_number = just.str.extract(r"\b(\d{4})\b", expand=False)
print("first four-digit number = heading:", round((first_number == decisions["heading"]).mean(), 2))

with_just = decisions["description"] + " " + just
past = (decisions["start_date"].dt.year <= 2021).to_numpy()
tr, va = train_test_split(np.flatnonzero(past), test_size=0.2, random_state=0)
new = np.flatnonzero(~past)
rows = []
for name, X in [("description only", decisions["description"]), ("+ justification", with_just)]:
    m = make_model().fit(X.iloc[tr], decisions["heading"].iloc[tr])
    rows.append({"inputs": name,
                 "validation (random, 2017-2021)": accuracy_score(decisions["heading"].iloc[va], m.predict(X.iloc[va])),
                 "in use (2022-2023, description only)": accuracy_score(
                     decisions["heading"].iloc[new], m.predict(decisions["description"].iloc[new]))})
pd.DataFrame(rows).round(3)

justification names the heading: 0.7


first four-digit number = heading: 0.64


,inputs,"validation (random, 2017-2021)","in use (2022-2023, description only)"
0,description only,0.806,0.757
1,+ justification,0.959,0.713


**Question.** Which number would a careless notebook report, and which number will the leaderboard show? Repeat the experiment with `keywords` instead of the justification.

## 5. Improve the classifier

Evaluate every change on the validation years, and report the subgroup without a quoted heading number separately: that is where the model has to understand the description.

In [10]:
no_number = ~has_number.to_numpy()


def evaluate(name, **tfidf_kw):
    p = make_model(**tfidf_kw).fit(train["description"], train["heading"]).predict(valid["description"])
    return {"change": name, "accuracy": accuracy_score(valid["heading"], p),
            "macro_F1": f1_score(valid["heading"], p, average="macro"),
            "accuracy without number": accuracy_score(valid["heading"][no_number], p[no_number])}


results = [evaluate("baseline (word unigrams)"),
           evaluate("min_df=1", min_df=1)]
pd.DataFrame(results).round(3)

,change,accuracy,macro_F1,accuracy without number
0,baseline (word unigrams),0.772,0.511,0.705
1,min_df=1,0.774,0.517,0.707
